# Module 3: LLM Fine-Tuning with QLoRA

**Alwajeez Tech -- Agentic AI Training Program**

---

## Overview

In this lab, you will learn how to **fine-tune** a large language model (LLM) to specialize it for the
Alwajeez Tech domain -- specifically, knowledge about **ZATCA compliance**, **Saudi VAT regulations**,
and the **Agentic AI System** architecture (FastAPI + Ollama + n8n + Redis).

### What is Fine-Tuning?

Fine-tuning is the process of taking a **pre-trained** model (one that has already learned general
language understanding from billions of tokens) and continuing its training on a **smaller, domain-specific
dataset**. This teaches the model to produce outputs that are more relevant, accurate, and aligned with
your specific use case.

Think of it like this: a pre-trained model is like a university graduate with broad knowledge. Fine-tuning
is like giving that graduate specialized on-the-job training at your company.

### What is LoRA?

**LoRA (Low-Rank Adaptation)** is a parameter-efficient fine-tuning technique. Instead of updating
**all** model weights (which would require massive GPU memory and storage), LoRA freezes the original
model and injects small **trainable adapter matrices** into specific layers (typically the attention
and MLP projection layers). Key concepts:

- **Rank (`r`)**: The rank of the adapter matrices. Higher rank = more capacity but more parameters.
  Typical values: 8, 16, 32, 64.
- **Alpha (`lora_alpha`)**: A scaling factor for the LoRA updates. Usually set to 2x the rank.
- **Target Modules**: Which layers to attach LoRA adapters to (e.g., `q_proj`, `v_proj`, `k_proj`).

The result: you train only **~1-10%** of the total parameters, but achieve performance close to
full fine-tuning.

### What is QLoRA?

**QLoRA (Quantized LoRA)** combines LoRA with **4-bit quantization**. The base model weights are
loaded in 4-bit precision (using the NF4 data type) instead of the usual 16-bit or 32-bit. This
**dramatically reduces memory usage** -- a 7B parameter model that normally requires ~14GB of VRAM
can fit in ~4GB. The LoRA adapters are still trained in higher precision (float16), so training
quality is preserved.

### What We Will Do in This Lab

1. Prepare an instruction-tuning dataset with domain-specific examples
2. Load TinyLlama (1.1B parameters) with 4-bit quantization
3. Attach LoRA adapters and fine-tune with SFTTrainer
4. Evaluate before vs. after fine-tuning
5. Merge the adapter, export to GGUF, and import into Ollama

### Requirements

- **Runtime**: Google Colab with **T4 GPU** (free tier works!)
- **Time**: ~30-45 minutes total
- **Libraries**: transformers, peft, trl, bitsandbytes, datasets

---

## Section 1: Setup (Google Colab)

First, let's verify we have GPU access and install the required libraries.

**Important**: Make sure your Colab runtime is set to **T4 GPU**:
- Go to `Runtime` > `Change runtime type` > Select `T4 GPU` > `Save`

In [1]:
# Google Colab Setup
# Check that we have a GPU available and install required packages
!nvidia-smi
!pip install -q torch transformers peft datasets accelerate bitsandbytes trl

Tue Feb 24 05:54:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.83                 Driver Version: 581.83         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce MX130         WDDM  |   00000000:02:00.0 Off |                  N/A |
| N/A    0C    P0            N/A  /  200W |       0MiB /   2048MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Verify GPU availability and print library versions
import torch

print("=" * 60)
print("GPU Information")
print("=" * 60)
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Please switch to a T4 GPU runtime.")
    print("Go to Runtime > Change runtime type > T4 GPU")

print()
print("=" * 60)
print("Library Versions")
print("=" * 60)
import transformers, peft, datasets, trl
print(f"Transformers: {transformers.__version__}")
print(f"PEFT: {peft.__version__}")
print(f"Datasets: {datasets.__version__}")
print(f"TRL: {trl.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")

GPU Information
CUDA available: False
Go to Runtime > Change runtime type > T4 GPU

Library Versions


d:\AI\agentic-ai-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers: 5.2.0
PEFT: 0.18.1
Datasets: 4.5.0
TRL: 0.28.0
PyTorch: 2.10.0+cpu
CUDA version: None


: 

**Expected Output**: You should see a Tesla T4 GPU with ~15 GB VRAM. If you see "No GPU detected",
change your runtime type as described above.

---

## Section 2: Prepare Dataset

### Understanding Instruction-Tuning Data Format

For fine-tuning, we use an **instruction-tuning** dataset format. Each example has three fields:

| Field | Description | Example |
|-------|-------------|---------|
| `instruction` | The task or question for the model | "What is ZATCA?" |
| `input` | Optional additional context | "" (empty) or specific context |
| `output` | The desired model response | "ZATCA stands for..." |

Our dataset covers three main domains:
1. **ZATCA & VAT** -- Saudi regulatory compliance knowledge
2. **System Architecture** -- How the Agentic AI System works
3. **Tool Usage & Operations** -- How to use tools, deploy, and maintain the system

**Note**: In production, you would want hundreds or thousands of examples. For this lab,
we use a small dataset of 20 examples to demonstrate the process. Even with small datasets,
fine-tuning can teach the model to adopt the right tone, format, and domain vocabulary.

In [ ]:
# Define our instruction-tuning dataset inline
# Each example teaches the model something specific about our domain

training_examples = [
    {
        "instruction": "What is ZATCA and what does it regulate?",
        "input": "",
        "output": "ZATCA stands for Zakat, Tax and Customs Authority. It is the Saudi Arabian government body responsible for collecting zakat (Islamic tax), enforcing tax regulations including VAT, and managing customs operations. ZATCA mandates electronic invoicing (e-invoicing) for all businesses in Saudi Arabia through a phased approach: Phase 1 requires generating invoices in a specific XML/PDF format, while Phase 2 requires real-time integration with ZATCA's Fatoora platform for invoice clearance and reporting."
    },
    {
        "instruction": "What is the current VAT rate in Saudi Arabia?",
        "input": "",
        "output": "The standard VAT rate in Saudi Arabia is 15%, which was increased from 5% to 15% in July 2020. Certain supplies are zero-rated (0%) including exports of goods and services, international transportation, and supplies of certain metals like gold and silver for investment. Some supplies are exempt from VAT entirely, such as certain financial services, residential real estate rentals, and life insurance."
    },
    {
        "instruction": "Explain the architecture of the Agentic AI System.",
        "input": "",
        "output": "The Agentic AI System consists of four main components: 1) FastAPI Server (port 8000) - the main API layer with OpenAI-compatible endpoints including /v1/chat/completions for direct LLM inference and /v1/agent/run for ReAct agent execution. 2) Ollama (port 11434) - the local LLM backend running models like llama3.2:3b for inference without cloud dependencies. 3) n8n (port 5678) - a workflow automation engine connected via webhooks for building business process automations. 4) Redis (port 6379) - provides caching and message queuing. All services are deployed via Docker Compose on a shared network."
    },
    {
        "instruction": "How does the ReAct agent loop work?",
        "input": "",
        "output": "The ReAct agent implements a Thought-Action-Observation loop. First, the agent receives a task and thinks about how to approach it (THOUGHT). Then it decides to call a tool with specific parameters (ACTION), formatted as ACTION: tool_name({\"param\": \"value\"}). The tool executes and returns a result (OBSERVATION). The agent analyzes the observation and decides whether to take another action or provide a FINAL ANSWER. This loop continues for up to MAX_ITERATIONS (default: 10) until the agent reaches a conclusion."
    },
    {
        "instruction": "What tools are available in the Agentic AI System?",
        "input": "",
        "output": "The system includes seven built-in tools: 1) calculator - evaluates mathematical expressions safely. 2) web_search - searches the internet for current information. 3) get_current_time - returns the current date and time in any timezone. 4) file_operations - reads, writes, and lists files in designated directories. 5) http_request - makes GET/POST/PUT/DELETE requests to external APIs. 6) zatca_compliance - validates and generates ZATCA-compliant e-invoices. 7) send_webhook - sends data to n8n webhooks for workflow automation."
    },
    {
        "instruction": "How do I use the calculator tool?",
        "input": "Calculate 15% VAT on 1000 SAR",
        "output": "To calculate 15% VAT on 1000 SAR using the calculator tool, you would call: ACTION: calculator({\"expression\": \"1000 * 0.15\"}). The result would be 150.0 SAR. The total invoice amount including VAT would be 1150 SAR (1000 + 150). The calculator tool accepts mathematical expressions with operators +, -, *, /, parentheses, and decimal numbers."
    },
    {
        "instruction": "What is the difference between the chat and agent endpoints?",
        "input": "",
        "output": "The chat endpoint (POST /v1/chat/completions) provides direct LLM inference - it forwards messages to the Ollama model and returns the response without any tool calling or reasoning loop. The agent endpoint (POST /v1/agent/run) implements the full ReAct loop where the AI reasons about the task, decides which tools to call, executes them, observes results, and iterates until reaching a final answer. Use the chat endpoint for simple Q&A and the agent endpoint for complex tasks that require tool use, multi-step reasoning, or external data access."
    },
    {
        "instruction": "How do I deploy the Agentic AI System?",
        "input": "",
        "output": "The system is deployed using Docker Compose. Steps: 1) Clone the repository. 2) Ensure Docker and Docker Compose are installed. 3) For GPU support, install NVIDIA Container Toolkit. 4) Run 'docker-compose up -d' to start all services. 5) The model-loader service will automatically pull the llama3.2:3b model on first startup. 6) Access the API at http://localhost:8000, n8n at http://localhost:5678. Verify deployment with GET http://localhost:8000/health which checks Ollama connectivity and returns system status."
    },
    {
        "instruction": "What are ZATCA e-invoicing Phase 1 and Phase 2 requirements?",
        "input": "",
        "output": "ZATCA e-invoicing has two phases. Phase 1 (Generation Phase, effective Dec 4, 2021): All businesses must generate tax invoices and simplified tax invoices in electronic format. Invoices must include a QR code, UUID, and be in XML format. Phase 2 (Integration Phase, started Jan 1, 2023, rolled out in waves): Businesses must integrate their e-invoicing systems with ZATCA's Fatoora platform for real-time clearance. Standard tax invoices (B2B) require clearance before being sent to buyers. Simplified invoices (B2C) must be reported within 24 hours."
    },
    {
        "instruction": "How does n8n integrate with the AI agent?",
        "input": "",
        "output": "n8n integrates with the Agentic AI System in two directions: 1) n8n-to-Agent: n8n workflows can trigger the AI agent by sending HTTP requests to POST /v1/agent/run or POST /webhook/n8n. The webhook endpoint accepts events with an 'agent_task' type that automatically creates and runs an agent task. 2) Agent-to-n8n: The agent can push data to n8n workflows using the send_webhook tool, which sends POST requests to http://localhost:5678/webhook/{webhook_name}. This enables building automation pipelines where AI reasoning triggers business workflows."
    },
    {
        "instruction": "What model should I use for tool calling?",
        "input": "",
        "output": "For tool calling with the Agentic AI System, the recommended model is llama3.2:3b, which is the default model. It offers a good balance between performance and resource usage, supporting instruction following and structured output needed for tool calling. For better accuracy on complex tasks, you can upgrade to llama3.1:8b. For code-specific tasks, codellama is recommended. The system uses a text-based tool calling format (ACTION: tool_name({...})) rather than native function calling, so most instruction-following models work."
    },
    {
        "instruction": "How do I check system health?",
        "input": "",
        "output": "Send a GET request to http://localhost:8000/health. The endpoint returns a JSON response with: 'status' (server health), 'ollama' (Ollama connectivity - 'healthy', 'unhealthy', or 'unreachable'), 'model' (configured model name), and 'timestamp' (current server time). You can also check individual services: Ollama at http://localhost:11434/api/tags, n8n at http://localhost:5678/healthz, and Redis with 'redis-cli ping'. Docker health checks are configured for all services with 30-second intervals."
    },
    {
        "instruction": "How do I add a custom tool to the system?",
        "input": "",
        "output": "To add a custom tool: 1) Define a ToolDefinition in the BUILTIN_TOOLS dictionary in agentic_server.py with name, description, and JSON Schema parameters. 2) Add a handler method in the ToolExecutor class (e.g., _my_custom_tool). 3) Add an elif branch in the ToolExecutor.execute() method to route to your handler. 4) Restart the server. The agent will automatically see the new tool in its system prompt and can decide to use it during the ReAct loop. The tool description should clearly explain when and how the agent should use it."
    },
    {
        "instruction": "What is the maximum number of agent iterations?",
        "input": "",
        "output": "The default maximum number of agent iterations is 10, configured via the MAX_ITERATIONS environment variable. Each iteration consists of one LLM call that produces a thought, optionally followed by a tool execution. If the agent doesn't reach a FINAL ANSWER within the maximum iterations, it returns with status 'max_iterations_reached'. You can override this per-request by setting max_iterations in the AgentTask payload. For simple tasks 3-5 iterations are typical; complex multi-tool tasks may need 7-10."
    },
    {
        "instruction": "Explain the file safety mechanism in the system.",
        "input": "",
        "output": "The ToolExecutor's file_operations method implements safety checks to prevent unauthorized file access. It only allows operations in specific directories: ./data, ./output, and ./temp. Any file path that doesn't start with one of these allowed directories is rejected with an error. Read operations are limited to returning the first 5000 characters to prevent memory issues. Write operations automatically create parent directories if they don't exist. These restrictions ensure the agent cannot access sensitive system files or configuration data."
    },
    {
        "instruction": "What is QLoRA and how is it used for fine-tuning?",
        "input": "",
        "output": "QLoRA (Quantized Low-Rank Adaptation) is an efficient fine-tuning technique that combines quantization with LoRA adapters. The base model is loaded in 4-bit quantization using BitsAndBytesConfig (nf4 type with double quantization), reducing memory usage from ~4GB to ~1GB for a 1B parameter model. LoRA adapters (small trainable matrices) are attached to the model's attention layers (q_proj, k_proj, v_proj, o_proj) and MLP layers (gate_proj, up_proj, down_proj). Only these adapters are trained while the base model stays frozen, requiring ~10% of the parameters of full fine-tuning."
    },
    {
        "instruction": "How do I generate a ZATCA-compliant e-invoice using the system?",
        "input": "Create an invoice for 5000 SAR service",
        "output": "To generate a ZATCA-compliant e-invoice, use the agent endpoint with the zatca_compliance tool. Send a POST request to /v1/agent/run with the task: 'Generate a ZATCA-compliant invoice for a 5000 SAR service.' The agent will call ACTION: zatca_compliance({\"action\": \"generate\", \"amount\": 5000, \"vat_rate\": 0.15, \"type\": \"simplified\"}) which returns an XML invoice with a QR code, UUID, VAT amount of 750 SAR, and total of 5750 SAR. The invoice conforms to ZATCA Phase 2 requirements."
    },
    {
        "instruction": "What is the Fatoora platform?",
        "input": "",
        "output": "Fatoora is ZATCA's e-invoicing platform that serves as the central hub for electronic invoice processing in Saudi Arabia. In Phase 2 of ZATCA's e-invoicing mandate, all businesses must connect their invoicing systems to Fatoora. For B2B standard tax invoices, Fatoora performs real-time clearance -- the invoice is validated and approved before it can be sent to the buyer. For B2C simplified invoices, businesses must report them to Fatoora within 24 hours. Fatoora ensures invoice integrity through cryptographic stamps and hash validation."
    },
    {
        "instruction": "How do I configure the system to use a different LLM model?",
        "input": "",
        "output": "To change the LLM model, update the MODEL_NAME environment variable in the docker-compose.yml file or .env file. For example, set MODEL_NAME=llama3.1:8b for a larger model. Then restart the services with 'docker-compose up -d'. The model-loader service will automatically pull the new model from Ollama's registry. You can also pull models manually with 'docker exec ollama ollama pull model_name'. Available models include llama3.2:3b (default, 3B params), llama3.1:8b (better accuracy), and codellama (code tasks)."
    },
    {
        "instruction": "What are the best practices for writing training data for fine-tuning?",
        "input": "",
        "output": "Best practices for fine-tuning training data: 1) Consistency -- all outputs should follow the same format, tone, and level of detail. 2) Accuracy -- every fact must be correct; the model will memorize errors. 3) Diversity -- cover different question types: factual, procedural, comparison, troubleshooting. 4) Specificity -- include domain-specific terms, numbers, and proper nouns (e.g., 'ZATCA', '15% VAT', 'port 8000'). 5) Completeness -- outputs should be self-contained and not reference external context. 6) Volume -- aim for at least 100-500 examples for meaningful improvement; 1000+ for strong results."
    }
]

print(f"Total training examples created: {len(training_examples)}")

In [ ]:
import json

# Split: first 15 for training, last 5 for evaluation
train_data = training_examples[:15]
eval_data = training_examples[15:]

# Save training data as JSONL (one JSON object per line)
with open("training_data.jsonl", "w", encoding="utf-8") as f:
    for example in train_data:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

# Save evaluation data as JSONL
with open("eval_data.jsonl", "w", encoding="utf-8") as f:
    for example in eval_data:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

print(f"Training examples saved: {len(train_data)} -> training_data.jsonl")
print(f"Evaluation examples saved: {len(eval_data)} -> eval_data.jsonl")
print()

# Verify the files were written correctly
with open("training_data.jsonl", "r", encoding="utf-8") as f:
    lines = f.readlines()
    print(f"Verification: training_data.jsonl has {len(lines)} lines")
    print(f"First line preview: {lines[0][:100]}...")

In [ ]:
# Print dataset statistics
import json

all_examples = training_examples

instruction_lengths = [len(ex["instruction"]) for ex in all_examples]
output_lengths = [len(ex["output"]) for ex in all_examples]
input_counts = sum(1 for ex in all_examples if ex.get("input", ""))

print("=" * 60)
print("Dataset Statistics")
print("=" * 60)
print(f"Total examples:          {len(all_examples)}")
print(f"  Training set:          {len(train_data)}")
print(f"  Evaluation set:        {len(eval_data)}")
print(f"  With input context:    {input_counts}")
print(f"  Without input context: {len(all_examples) - input_counts}")
print()
print(f"Avg instruction length:  {sum(instruction_lengths) / len(instruction_lengths):.0f} chars")
print(f"Min instruction length:  {min(instruction_lengths)} chars")
print(f"Max instruction length:  {max(instruction_lengths)} chars")
print()
print(f"Avg output length:       {sum(output_lengths) / len(output_lengths):.0f} chars")
print(f"Min output length:       {min(output_lengths)} chars")
print(f"Max output length:       {max(output_lengths)} chars")

---

## Section 3: Load & Format Dataset

### Chat Template Formatting

Before training, we need to convert our instruction/output pairs into the **chat template format**
that TinyLlama expects. TinyLlama uses a format with special tokens:

```
<|system|>
System message here</s>
<|user|>
User message here</s>
<|assistant|>
Assistant response here</s>
```

We set the system message to define the model's persona as **Alwajeez AI**, an expert in ZATCA,
Saudi VAT, and the Agentic AI System.

The `format_instruction` function below converts each training example into this format. If the
example has an `input` field (additional context), we append it to the user message.

In [ ]:
from datasets import load_dataset

# Load our JSONL files into a HuggingFace Dataset object
dataset = load_dataset("json", data_files={
    "train": "training_data.jsonl",
    "eval": "eval_data.jsonl"
})

print(f"Dataset loaded: {dataset}")
print(f"Train columns: {dataset['train'].column_names}")
print(f"Train size: {len(dataset['train'])}")
print(f"Eval size: {len(dataset['eval'])}")


def format_instruction(example):
    """
    Convert an instruction/input/output example into TinyLlama's chat format.
    
    This function is applied to every example in the dataset using .map().
    The resulting 'text' field is what the model will be trained on.
    """
    system_msg = (
        "You are Alwajeez AI, a helpful assistant specialized in "
        "ZATCA compliance, Saudi VAT regulations, and the Agentic AI System."
    )
    
    if example.get("input", ""):
        text = (
            f"<|system|>\n{system_msg}</s>\n"
            f"<|user|>\n{example['instruction']}\n{example['input']}</s>\n"
            f"<|assistant|>\n{example['output']}</s>"
        )
    else:
        text = (
            f"<|system|>\n{system_msg}</s>\n"
            f"<|user|>\n{example['instruction']}</s>\n"
            f"<|assistant|>\n{example['output']}</s>"
        )
    return {"text": text}


# Apply formatting to all examples
dataset = dataset.map(format_instruction)
print("\nDataset formatted successfully!")
print(f"New columns: {dataset['train'].column_names}")

In [ ]:
# Print a sample formatted example to verify the formatting
print("=" * 60)
print("Sample Formatted Training Example")
print("=" * 60)
print(dataset["train"][0]["text"])
print()
print("=" * 60)
print("Sample with Input Context (example index 5)")
print("=" * 60)
print(dataset["train"][5]["text"])

---

## Section 4: Load Base Model with QLoRA

### Understanding BitsAndBytesConfig

We use the `BitsAndBytesConfig` class to configure **4-bit quantization** for the base model.
Here is what each parameter does:

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `load_in_4bit` | `True` | Load model weights in 4-bit precision |
| `bnb_4bit_quant_type` | `"nf4"` | Use **NormalFloat4** -- a data type optimized for normally-distributed weights |
| `bnb_4bit_compute_dtype` | `float16` | Use FP16 for computation (faster on T4) |
| `bnb_4bit_use_double_quant` | `True` | Quantize the quantization constants too (saves ~0.4 bits/param) |

### Why TinyLlama?

We use **TinyLlama-1.1B-Chat** for this lab because:
- It is small enough to fine-tune on a free T4 GPU in Colab
- It already supports the chat template format we need
- It demonstrates the full QLoRA workflow that scales to larger models
- In production, you would use the same process with a larger model (e.g., Llama 3.1 8B)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# The base model to fine-tune
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Configure 4-bit quantization (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # Load weights in 4-bit
    bnb_4bit_quant_type="nf4",            # NormalFloat4 quantization
    bnb_4bit_compute_dtype=torch.float16,  # Compute in FP16
    bnb_4bit_use_double_quant=True,        # Double quantization for extra savings
)

print(f"Loading model: {model_name}")
print("This may take 1-2 minutes on first download...")

# Load the model with quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",  # Automatically place layers on GPU/CPU
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Set padding token
tokenizer.padding_side = "right"            # Pad on the right side

print(f"\nModel loaded successfully!")
print(f"Model parameters: {model.num_parameters():,}")
print(f"Model dtype: {model.dtype}")
print(f"Tokenizer vocab size: {tokenizer.vocab_size:,}")

# Check GPU memory usage after loading
if torch.cuda.is_available():
    mem_allocated = torch.cuda.memory_allocated() / 1e9
    mem_reserved = torch.cuda.memory_reserved() / 1e9
    print(f"GPU memory allocated: {mem_allocated:.2f} GB")
    print(f"GPU memory reserved: {mem_reserved:.2f} GB")

In [ ]:
# Test the BASE model (before fine-tuning) on a domain-specific question
# This gives us a baseline to compare against after fine-tuning

prompt = (
    "<|system|>\n"
    "You are a helpful assistant.</s>\n"
    "<|user|>\n"
    "What is ZATCA and what are its e-invoicing phases?</s>\n"
    "<|assistant|>\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

print("Generating response from BASE model (before fine-tuning)...")
print("=" * 60)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
    )

base_response = tokenizer.decode(outputs[0], skip_special_tokens=False)
print("Base model response:")
print(base_response)
print("=" * 60)
print()
print("NOTE: The base model likely gives a generic or incorrect answer about ZATCA.")
print("After fine-tuning, it should provide specific, accurate information.")

---

## Section 5: Configure LoRA

### LoRA Parameters Explained

Now we configure the LoRA adapters that will be attached to the frozen base model.

| Parameter | Value | Explanation |
|-----------|-------|-------------|
| `r` | 16 | **Rank** of the adapter matrices. Determines the capacity of the adaptation. r=16 is a good balance between performance and efficiency. |
| `lora_alpha` | 32 | **Scaling factor**. The effective learning rate of LoRA is scaled by `alpha/r`. Setting alpha=2*r is a common heuristic. |
| `target_modules` | (see below) | Which linear layers get LoRA adapters. We target all attention projections (q, k, v, o) and MLP projections (gate, up, down). |
| `lora_dropout` | 0.05 | Small dropout for regularization, prevents overfitting on our small dataset. |
| `bias` | "none" | Do not train bias terms (saves parameters). |
| `task_type` | "CAUSAL_LM" | We are fine-tuning for causal language modeling (text generation). |

### Target Modules

For a Transformer model, the key linear layers are:
- **Attention**: `q_proj` (query), `k_proj` (key), `v_proj` (value), `o_proj` (output)
- **MLP/FFN**: `gate_proj`, `up_proj`, `down_proj`

Targeting all of these gives the best fine-tuning quality. If memory is tight, you can
start with just `q_proj` and `v_proj` as a minimum.

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

# Step 1: Prepare the quantized model for training
# This handles gradient checkpointing and layer norm casting for stable training
model = prepare_model_for_kbit_training(model)

# Step 2: Define the LoRA configuration
lora_config = LoraConfig(
    r=16,                      # Rank of the adapter
    lora_alpha=32,             # Scaling factor (2 * r)
    target_modules=[           # Layers to attach LoRA to
        "q_proj",   # Query projection (attention)
        "k_proj",   # Key projection (attention)
        "v_proj",   # Value projection (attention)
        "o_proj",   # Output projection (attention)
        "gate_proj", # Gate projection (MLP)
        "up_proj",   # Up projection (MLP)
        "down_proj", # Down projection (MLP)
    ],
    lora_dropout=0.05,         # Dropout for regularization
    bias="none",               # Don't train biases
    task_type="CAUSAL_LM",     # Task type: causal language modeling
)

# Step 3: Wrap the model with LoRA adapters
model = get_peft_model(model, lora_config)

# Print the number of trainable parameters
print("=" * 60)
print("Model Parameter Summary")
print("=" * 60)
model.print_trainable_parameters()
print()
print("Note: Only a small fraction of parameters are trainable (the LoRA adapters).")
print("The base model weights remain frozen throughout training.")

You should see something like:
```
trainable params: 6,815,744 || all params: 1,108,892,672 || trainable%: 0.6147
```

This means we are only training **~0.6%** of the model's parameters!

---

## Section 6: Fine-Tune with SFTTrainer

### Understanding SFTTrainer

**SFTTrainer** (Supervised Fine-Tuning Trainer) from the `trl` library is a wrapper around
HuggingFace's `Trainer` that simplifies instruction tuning. It handles:

- Tokenization of the `text` field in our dataset
- Sequence packing and padding
- Loss computation (standard cross-entropy on next-token prediction)
- Gradient accumulation and mixed-precision training

### Training Hyperparameters Explained

| Parameter | Value | Explanation |
|-----------|-------|-------------|
| `num_train_epochs` | 3 | Number of passes through the training data. With small datasets, 3-5 epochs is common. |
| `per_device_train_batch_size` | 4 | Number of examples per GPU per step. Limited by VRAM. |
| `gradient_accumulation_steps` | 4 | Accumulate gradients over 4 steps before updating. Effective batch size = 4 * 4 = 16. |
| `learning_rate` | 2e-4 | Standard for LoRA fine-tuning. Higher than full fine-tuning (1e-5) because we have fewer parameters. |
| `lr_scheduler_type` | "cosine" | Learning rate decays following a cosine curve. Smoother than linear decay. |
| `warmup_ratio` | 0.1 | Spend 10% of training warming up the learning rate from 0. Prevents early instability. |
| `fp16` | True | Use FP16 mixed precision for faster training on T4 GPU. |
| `max_seq_length` | 512 | Maximum token length per example. Longer sequences use more memory. |

In [ ]:
from trl import SFTTrainer, SFTConfig

# Define training arguments
training_args = SFTConfig(
    output_dir="./results",                   # Where to save checkpoints
    num_train_epochs=3,                        # Number of training epochs
    per_device_train_batch_size=4,             # Batch size per GPU
    gradient_accumulation_steps=4,             # Effective batch size = 4 * 4 = 16
    learning_rate=2e-4,                        # Learning rate for LoRA
    lr_scheduler_type="cosine",                # Cosine learning rate decay
    warmup_ratio=0.1,                          # 10% warmup
    fp16=True,                                 # Mixed precision training
    logging_steps=5,                           # Log every 5 steps
    save_strategy="epoch",                     # Save checkpoint each epoch
    eval_strategy="epoch",                     # Evaluate each epoch
    max_seq_length=512,                        # Max sequence length
    dataset_text_field="text",                 # Column with formatted text
    report_to="none",                          # Disable wandb/tensorboard
)

# Create the SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    tokenizer=tokenizer,
)

print("Trainer created successfully!")
print(f"Training examples: {len(dataset['train'])}")
print(f"Evaluation examples: {len(dataset['eval'])}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print()
print("Starting training... (this may take 5-10 minutes on a T4 GPU)")
print("=" * 60)

# Train!
trainer.train()

In [ ]:
# Plot training loss curve
import matplotlib.pyplot as plt

logs = trainer.state.log_history

# Extract training losses (steps where 'loss' key exists)
train_losses = [(l["step"], l["loss"]) for l in logs if "loss" in l]
# Extract evaluation losses (steps where 'eval_loss' key exists)
eval_losses = [(l["step"], l["eval_loss"]) for l in logs if "eval_loss" in l]

fig, ax = plt.subplots(1, 1, figsize=(10, 5))

if train_losses:
    steps, losses = zip(*train_losses)
    ax.plot(steps, losses, label="Training Loss", color="blue", marker="o", markersize=3)

if eval_losses:
    steps, losses = zip(*eval_losses)
    ax.plot(steps, losses, label="Eval Loss", color="red", marker="s", markersize=6)

ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("Training & Evaluation Loss -- Alwajeez QLoRA Fine-Tuning")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print final metrics
if train_losses:
    print(f"Initial training loss: {train_losses[0][1]:.4f}")
    print(f"Final training loss:   {train_losses[-1][1]:.4f}")
if eval_losses:
    print(f"Final eval loss:       {eval_losses[-1][1]:.4f}")

In [ ]:
# Save the LoRA adapter (NOT the full model -- just the small adapter weights)
adapter_path = "./alwajeez-lora-adapter"
trainer.save_model(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"LoRA adapter saved to: {adapter_path}")

# Show what files were saved
import os
print("\nSaved files:")
for f in os.listdir(adapter_path):
    size_mb = os.path.getsize(os.path.join(adapter_path, f)) / 1e6
    print(f"  {f}: {size_mb:.2f} MB")

**Note**: The adapter is very small (typically ~25-30 MB) compared to the full model (~2 GB).
This is the key advantage of LoRA -- you can store many specialized adapters without duplicating
the entire base model.

---

## Section 7: Evaluate Fine-Tuned vs Base

### Evaluation Approach

To evaluate the impact of fine-tuning, we compare the fine-tuned model's responses against
what we observed from the base model in Section 4. We use four domain-specific test prompts
that cover different aspects of our training data:

1. **ZATCA knowledge** -- Tests regulatory domain expertise
2. **System architecture** -- Tests technical knowledge of our system
3. **Tool usage** -- Tests understanding of the agent's capabilities
4. **VAT calculation** -- Tests practical domain task performance

We look for:
- **Specificity**: Does the model mention specific names, numbers, and terms from our domain?
- **Accuracy**: Are the facts correct (e.g., 15% VAT, port 8000, Phase 1 vs Phase 2)?
- **Completeness**: Does it cover the key points?

In [ ]:
# Define test prompts for evaluation
test_prompts = [
    {
        "name": "ZATCA Knowledge",
        "prompt": "What is ZATCA and what are its e-invoicing phases?",
    },
    {
        "name": "System Architecture",
        "prompt": "Explain the architecture of the Agentic AI System and its main components.",
    },
    {
        "name": "Tool Usage",
        "prompt": "What tools are available in the Agentic AI System and how does the ReAct loop use them?",
    },
    {
        "name": "VAT Calculation",
        "prompt": "What is the VAT rate in Saudi Arabia and how do I calculate VAT on a 1000 SAR invoice?",
    },
]

print(f"Defined {len(test_prompts)} test prompts for evaluation.")
for i, tp in enumerate(test_prompts):
    print(f"  {i+1}. [{tp['name']}] {tp['prompt']}")

In [ ]:
# Generate responses from the FINE-TUNED model

system_msg = (
    "You are Alwajeez AI, a helpful assistant specialized in "
    "ZATCA compliance, Saudi VAT regulations, and the Agentic AI System."
)

finetuned_responses = []

for tp in test_prompts:
    prompt = (
        f"<|system|>\n{system_msg}</s>\n"
        f"<|user|>\n{tp['prompt']}</s>\n"
        f"<|assistant|>\n"
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
        )
    
    # Decode and extract only the assistant's response
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
    # Extract text after the last <|assistant|> tag
    if "<|assistant|>" in full_text:
        response = full_text.split("<|assistant|>")[-1].strip()
        # Remove trailing </s> if present
        response = response.replace("</s>", "").strip()
    else:
        response = full_text
    
    finetuned_responses.append(response)
    print(f"Generated response for: {tp['name']}")

print("\nAll fine-tuned responses generated!")

In [ ]:
# Generate BASE model responses for comparison
# We reload the base model without LoRA adapters to get a fair comparison

print("Loading fresh base model for comparison...")
print("(This reloads the model without LoRA adapters)\n")

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

base_responses = []

for tp in test_prompts:
    prompt = (
        f"<|system|>\nYou are a helpful assistant.</s>\n"
        f"<|user|>\n{tp['prompt']}</s>\n"
        f"<|assistant|>\n"
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
        )
    
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
    if "<|assistant|>" in full_text:
        response = full_text.split("<|assistant|>")[-1].strip()
        response = response.replace("</s>", "").strip()
    else:
        response = full_text
    
    base_responses.append(response)
    print(f"Generated base response for: {tp['name']}")

print("\nAll base responses generated!")

# Free the base model from memory
del base_model
torch.cuda.empty_cache()

In [ ]:
# Print side-by-side comparison

for i, tp in enumerate(test_prompts):
    print("=" * 80)
    print(f"TEST {i+1}: {tp['name']}")
    print(f"PROMPT: {tp['prompt']}")
    print("=" * 80)
    print()
    print("--- BASE MODEL (before fine-tuning) ---")
    print(base_responses[i][:500])  # Truncate for readability
    print()
    print("--- FINE-TUNED MODEL (after QLoRA) ---")
    print(finetuned_responses[i][:500])  # Truncate for readability
    print()
    print()

print("=" * 80)
print("EVALUATION SUMMARY")
print("=" * 80)
print("Look for these improvements in the fine-tuned model:")
print("  1. Mentions specific terms: ZATCA, Fatoora, Phase 1, Phase 2")
print("  2. Correct numbers: 15% VAT, port 8000, port 11434")
print("  3. System components: FastAPI, Ollama, n8n, Redis")
print("  4. Tool names: calculator, web_search, zatca_compliance")
print("  5. Alwajeez-specific tone and domain language")

---

## Section 8: Merge & Export

### Why Merge?

During training, the LoRA adapter is a **separate** set of weights from the base model. For
deployment, we want a **single merged model** that combines both. The `merge_and_unload()` method:

1. Adds the LoRA adapter weights into the corresponding base model layers
2. Removes the PEFT wrapper, returning a standard HuggingFace model
3. The result is a standalone model with no adapter dependency

### GGUF Conversion

To use our fine-tuned model with **Ollama**, we need to convert it from HuggingFace format
to **GGUF** (GPT-Generated Unified Format). GGUF is the format used by llama.cpp and Ollama
for efficient CPU/GPU inference. The conversion also applies quantization (e.g., Q4_K_M)
to keep the model small and fast.

In [ ]:
from peft import AutoPeftModelForCausalLM
import torch

# Reload the adapter from disk (demonstrates how to load a saved adapter)
print("Loading LoRA adapter from ./alwajeez-lora-adapter ...")

model = AutoPeftModelForCausalLM.from_pretrained(
    "./alwajeez-lora-adapter",
    device_map="auto",
    torch_dtype=torch.float16,
)

# Merge the adapter with the base model
print("Merging LoRA adapter with base model...")
merged_model = model.merge_and_unload()

# Save the merged model
output_dir = "./alwajeez-merged"
merged_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"\nMerged model saved to: {output_dir}")

# Show saved files and their sizes
import os
total_size = 0
print("\nSaved files:")
for f in sorted(os.listdir(output_dir)):
    fpath = os.path.join(output_dir, f)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        total_size += size_mb
        print(f"  {f}: {size_mb:.2f} MB")
print(f"\nTotal size: {total_size:.1f} MB")

In [ ]:
# GGUF Conversion Steps
# ====================
# These commands are run in a terminal (not in this notebook).
# They convert the HuggingFace model to GGUF format for Ollama.

print("=" * 60)
print("GGUF Conversion Instructions")
print("=" * 60)
print()
print("Run these commands in a terminal (on your server or local machine):")
print()
print("# Step 1: Clone llama.cpp (the GGUF conversion tool)")
print("git clone https://github.com/ggerganov/llama.cpp")
print()
print("# Step 2: Install Python requirements for conversion")
print("cd llama.cpp")
print("pip install -r requirements.txt")
print()
print("# Step 3: Convert HuggingFace model to GGUF with Q4_K_M quantization")
print("python convert_hf_to_gguf.py ../alwajeez-merged \\")
print("    --outtype q4_k_m \\")
print("    --outfile model-finetuned.gguf")
print()
print("# Quantization options (from smallest to largest):")
print("#   q4_0    - Fastest, lowest quality")
print("#   q4_k_m  - Good balance (RECOMMENDED)")
print("#   q5_k_m  - Better quality, slightly larger")
print("#   q8_0    - Highest quality, largest size")
print()
print("# Expected output file size for TinyLlama with Q4_K_M: ~600-700 MB")

In [ ]:
# Show the Modelfile used for importing into Ollama
# The Modelfile is like a Dockerfile -- it defines how Ollama should load and configure the model.

modelfile_content = """# Modelfile for importing fine-tuned model into Ollama
# Usage: ollama create alwajeez-finetuned -f Modelfile

FROM ./model-finetuned.gguf

TEMPLATE \"\"\"<|system|>
{{ .System }}</s>
<|user|>
{{ .Prompt }}</s>
<|assistant|>
{{ .Response }}</s>\"\"\"

SYSTEM \"\"\"You are Alwajeez AI, a specialized assistant for Alwajeez Tech. You are an expert in:
- ZATCA (Zakat, Tax and Customs Authority) e-invoicing compliance
- Saudi Arabia VAT regulations (15% standard rate)
- The Agentic AI System architecture (FastAPI + Ollama + n8n + Redis)
- Tool usage and ReAct agent patterns
Provide accurate, specific, and actionable answers.\"\"\"

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER stop "<|user|>"
PARAMETER stop "</s>"
"""

print("=" * 60)
print("Ollama Modelfile")
print("=" * 60)
print(modelfile_content)
print("=" * 60)
print("Modelfile Explanation:")
print("=" * 60)
print("  FROM          - Path to the GGUF model file")
print("  TEMPLATE      - The chat template format (must match training format!)")
print("  SYSTEM        - Default system prompt baked into the model")
print("  PARAMETER     - Generation parameters:")
print("    temperature - Controls randomness (0=deterministic, 1=creative)")
print("    top_p       - Nucleus sampling threshold")
print("    stop        - Tokens that signal the model to stop generating")

---

## Section 9: Test in Ollama

### Importing into Ollama

Now that we have the GGUF model and a Modelfile, we can import the fine-tuned model into
Ollama and use it with our existing Agentic AI System (FastAPI server). This is the final
step that connects your fine-tuning work to production.

### Integration with the Agentic AI System

The Agentic AI System is designed to be model-agnostic. You can swap in the fine-tuned model
by simply changing the `MODEL_NAME` environment variable. The FastAPI server, ReAct agent,
and all tools will work with the new model without any code changes.

In [ ]:
# Steps to import the fine-tuned model into Ollama and test it

print("=" * 60)
print("Deploying Fine-Tuned Model to Ollama")
print("=" * 60)
print()
print("""Steps to import into Ollama:

1. Copy the GGUF model and Modelfile to your server:
   scp model-finetuned.gguf Modelfile user@server:/path/to/models/

2. Create the Ollama model from the Modelfile:
   ollama create alwajeez-finetuned -f Modelfile

3. Test the model directly with Ollama:
   ollama run alwajeez-finetuned "What is ZATCA?"

4. Update docker-compose.yml to use the fine-tuned model:
   environment:
     - MODEL_NAME=alwajeez-finetuned

5. Restart the services:
   docker-compose up -d

6. Test via the FastAPI API:
   curl http://localhost:8000/v1/chat/completions \\
     -H "Content-Type: application/json" \\
     -d '{"messages":[{"role":"user","content":"What is ZATCA?"}],"model":"alwajeez-finetuned"}'

7. Test via the Agent endpoint:
   curl http://localhost:8000/v1/agent/run \\
     -H "Content-Type: application/json" \\
     -d '{"task":"Calculate 15%% VAT on 5000 SAR and explain ZATCA compliance requirements"}'   

8. Verify health:
   curl http://localhost:8000/health
""")

In [ ]:
# Optional: Save to Google Drive for persistence
# (Colab files are deleted when the runtime disconnects)

# Uncomment the lines below to save your model to Google Drive:

# from google.colab import drive
# drive.mount('/content/drive')
#
# import shutil
# import os
#
# # Create destination directory
# dest_dir = "/content/drive/MyDrive/alwajeez-finetuned"
# os.makedirs(dest_dir, exist_ok=True)
#
# # Copy merged model
# print("Copying merged model to Google Drive...")
# shutil.copytree("./alwajeez-merged", f"{dest_dir}/alwajeez-merged", dirs_exist_ok=True)
#
# # Copy LoRA adapter (smaller, useful for later experiments)
# print("Copying LoRA adapter to Google Drive...")
# shutil.copytree("./alwajeez-lora-adapter", f"{dest_dir}/alwajeez-lora-adapter", dirs_exist_ok=True)
#
# # Copy training data
# print("Copying training data to Google Drive...")
# shutil.copy2("training_data.jsonl", f"{dest_dir}/training_data.jsonl")
# shutil.copy2("eval_data.jsonl", f"{dest_dir}/eval_data.jsonl")
#
# print(f"\nAll files saved to: {dest_dir}")
# print("Your model will persist across Colab sessions.")

---

## Summary

In this lab, you learned:

1. **What fine-tuning is** and why it matters for domain-specific AI applications
2. **How LoRA works** -- attaching small trainable adapter matrices to frozen model layers
3. **How QLoRA works** -- combining LoRA with 4-bit quantization to fine-tune on consumer GPUs
4. **How to prepare training data** in instruction-tuning format (instruction/input/output)
5. **How to format data** using the model's chat template (system/user/assistant)
6. **How to load a model with BitsAndBytesConfig** for 4-bit quantization
7. **How to configure LoRA** parameters (rank, alpha, target modules, dropout)
8. **How to train with SFTTrainer** and monitor training loss
9. **How to evaluate** fine-tuned vs. base model responses
10. **How to merge and export** the model to GGUF format for Ollama deployment

### Key Takeaways

- **QLoRA makes fine-tuning accessible**: A T4 GPU (free on Colab) is enough to fine-tune a 1B+ model
- **Small datasets can still help**: Even 15-20 examples teach domain vocabulary and response style
- **The pipeline is model-agnostic**: The same process works for Llama 3.1 8B, Mistral, Qwen, etc.
- **Ollama makes deployment easy**: Convert to GGUF, write a Modelfile, and `ollama create`

### Next Steps -- Module 4: DPO/RLHF Alignment

In the next module, we will learn about **preference optimization** techniques:

- **RLHF (Reinforcement Learning from Human Feedback)** -- Training a reward model from human preferences and using PPO to optimize the LLM
- **DPO (Direct Preference Optimization)** -- A simpler alternative that skips the reward model and directly optimizes from preference pairs
- **Building preference datasets** -- How to collect chosen/rejected response pairs
- **When to use SFT vs. DPO** -- SFT teaches knowledge; DPO teaches style, safety, and alignment

This will further refine our Alwajeez AI model to produce responses that are not just
accurate but also well-formatted, safe, and aligned with our company's communication style.